# Problem statement
## With thousands of movies and TV shows available on Netflix, users often find it difficult to choose what to watch. At the same time, analyzing such a large collection of content manually to understand trends, genres, ratings, and viewing options is time-consuming. Our project, Netflix Content Analyzer and Recommendation System, aims to solve this problem by analyzing Netflix data, presenting meaningful insights through graphs and visualizations, and recommending relevant movies or TV shows based on user preferences.

In [4]:
# Netflix Content Analyzer + Recommender  (simple version)
# Libraries: pandas, numpy, matplotlib, seaborn, requests, beautifulsoup4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup

sns.set_theme(style="whitegrid")

# ---------------- 1. LOAD DATA ----------------
nf = pd.read_csv("netflix_titles.csv")
ts = pd.read_csv("titles.csv")

# ---------------- 2. CLEAN NETFLIX DATA ----------------
nf["type"] = nf["type"].map({"Movie": "MOVIE", "TV Show": "SHOW"})   # same style as titles.csv
nf["country"] = nf["country"].fillna("Unknown")
nf["listed_in"] = nf["listed_in"].fillna("")
nf["date_added"] = pd.to_datetime(nf["date_added"].astype(str).str.strip(), errors="coerce")
nf["year_added"] = nf["date_added"].dt.year

# some rows have "74 min" etc. in the rating column -> keep only real ratings
valid = ["G", "PG", "PG-13", "R", "NC-17", "NR", "UR",
         "TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "TV-PG", "TV-14", "TV-MA"]
nf["rating"] = nf["rating"].where(nf["rating"].isin(valid))

# ---------------- 3. CLEAN titles.csv AND MERGE ----------------
ts["type"] = ts["type"].str.upper()
for col in ["imdb_score", "imdb_votes"]:
    ts[col] = pd.to_numeric(ts[col], errors="coerce")

nf["key"] = nf["title"].str.lower().str.strip()
ts["key"] = ts["title"].str.lower().str.strip()

# keep the most-voted row if a title appears twice
ts_small = (ts[["key", "type", "release_year", "genres", "imdb_score", "imdb_votes"]]
            .sort_values("imdb_votes", ascending=False)
            .drop_duplicates(["key", "type", "release_year"]))

# left merge -> ONLY Netflix titles are kept
df = nf.merge(ts_small, on=["key", "type", "release_year"], how="left")

# all genres in one lowercase text column (from both files)
df["all_genres"] = (df["listed_in"] + "," + df["genres"].fillna("")).str.lower()

# popularity rank by IMDb votes (only titles that have a score AND votes)
ok = df["imdb_score"].notna() & df["imdb_votes"].notna()
df["popularity_rank"] = np.nan
df.loc[ok, "popularity_rank"] = df.loc[ok, "imdb_votes"].rank(ascending=False, method="min")

COLS = ["popularity_rank", "title", "type", "release_year", "rating", "imdb_score", "imdb_votes"]


def show(table):
    table = table[COLS].copy()
    table["popularity_rank"] = table["popularity_rank"].astype("Int64")   # 1.0 -> 1
    table["imdb_votes"] = table["imdb_votes"].astype("Int64")
    print(table.to_string(index=False))


def ranked(table):
    """Keep only titles that have a popularity rank (score + votes available)."""
    return table.dropna(subset=["popularity_rank"])

In [5]:
# ---------------- 4. GENERAL ANALYSIS ----------------
def summary():
    print("Shape:", df.shape)
    print("Matched with IMDb data:", df["imdb_score"].notna().sum(), "titles")
    print("\nMissing values:\n", df.isnull().sum())


def movie_vs_show():
    print(df["type"].value_counts())
    sns.countplot(data=df, x="type")
    plt.title("Movies vs TV Shows")
    plt.show()


def top_genres():
    genres = nf["listed_in"].str.split(",").explode().str.strip()
    top = genres.value_counts().head(10)
    print(top)
    sns.barplot(x=top.values, y=top.index)
    plt.title("Top 10 Genres")
    plt.show()


def top_countries():
    countries = nf["country"].str.split(",").explode().str.strip()
    top = countries[countries != "Unknown"].value_counts().head(10)
    print(top)
    sns.barplot(x=top.values, y=top.index)
    plt.title("Top 10 Countries")
    plt.show()


def release_trend():
    yearly = df[df["release_year"] >= 2000].groupby(["release_year", "type"]).size().reset_index(name="count")
    sns.lineplot(data=yearly, x="release_year", y="count", hue="type", marker="o")
    plt.title("Titles by Release Year")
    plt.show()


def rating_chart():
    print(df["rating"].value_counts())
    sns.countplot(data=df, y="rating", order=df["rating"].value_counts().index)
    plt.title("Content Ratings")
    plt.show()


def added_over_time():
    yearly = df.groupby("year_added").size()
    yearly.plot(kind="line", marker="o")
    plt.title("Titles Added to Netflix per Year")
    plt.xlabel("Year added")
    plt.ylabel("Titles")
    plt.show()

In [6]:
# ---------------- 5. IMDb ANALYSIS ----------------
def rating_analysis():
    r = df["imdb_score"].dropna()
    print("Average IMDb rating:", round(r.mean(), 2))
    sns.histplot(r, bins=20, kde=True)
    plt.title("IMDb Rating Distribution")
    plt.show()


def highest_rated():
    pool = df[df["imdb_votes"] >= 1000]   # ignore titles with very few votes
    for t in ["MOVIE", "SHOW"]:
        print("\nHighest rated", t)
        show(pool[pool["type"] == t].sort_values("imdb_score", ascending=False).head(10))


def most_popular():
    for t in ["MOVIE", "SHOW"]:
        print("\nMost popular", t, "(by IMDb votes)")
        show(ranked(df[df["type"] == t]).sort_values("popularity_rank").head(10))


def top_by_genre():
    g = input("Enter genre (e.g. thriller): ").lower().strip()
    pool = df[df["all_genres"].str.contains(g, na=False)]
    for t in ["MOVIE", "SHOW"]:
        print("\nTop", t, "in", g)
        show(ranked(pool[pool["type"] == t]).sort_values("popularity_rank").head(10))


def top_by_country():
    c = input("Enter country (e.g. India): ").lower().strip()
    pool = df[df["country"].str.lower().str.contains(c, na=False)]
    print("\nMost popular titles from", c, "(rank = global rank among all titles)")
    show(ranked(pool).sort_values("popularity_rank").head(10))


In [7]:
# ---------------- 6. RECOMMENDER (with sub menu) ----------------
def recommend_by_preferences():
    print("Press Enter to skip any question.")
    genre = input("Genre (Thriller/Comedy/Romance/Action): ").lower().strip()
    years = input("Release year range (e.g. 2015-2025): ").strip()
    country = input("Country (India/South Korea/United States): ").lower().strip()
    min_rating = input("Minimum IMDb rating (e.g. 8): ").strip()
    content_rating = input("Content rating (PG-13/TV-MA): ").upper().strip()
    kind = input("Type - 1 Movies, 2 TV Shows, Enter for both: ").strip()

    r = df.copy()
    if genre:
        r = r[r["all_genres"].str.contains(genre, na=False)]
    if years:
        try:
            start, end = [int(y) for y in years.split("-")]
            r = r[r["release_year"].between(start, end)]
        except ValueError:
            print("Year range ignored (use format 2015-2025).")
    if country:
        r = r[r["country"].str.lower().str.contains(country, na=False)]
    if min_rating:
        try:
            r = r[r["imdb_score"] >= float(min_rating)]
        except ValueError:
            print("Rating ignored (enter a number).")
    if content_rating:
        r = r[r["rating"] == content_rating]
    if kind == "1":
        r = r[r["type"] == "MOVIE"]
    elif kind == "2":
        r = r[r["type"] == "SHOW"]

    if r.empty:
        print("\nNo titles match your preferences. Try fewer or wider filters.")
        return
    print("\n", len(r), "matches, ranked by popularity:\n")
    show(r.sort_values("popularity_rank"))


def show_with_people(r):
    """Print the ranked table plus director and main cast of each title."""
    top = r.sort_values("popularity_rank")
    show(top)
    print("\nDirector and main cast:")
    extra = top[["title", "director", "cast"]].copy()
    extra["director"] = extra["director"].fillna("Not listed")
    extra["cast"] = extra["cast"].fillna("Not listed").str.split(",").str[:3].str.join(",")
    print(extra.to_string(index=False))


def recommend_by_actor():
    name = input("Actor / Actress name (e.g. Shah Rukh Khan): ").lower().strip()
    if not name:
        print("Please enter a name.")
        return
    r = df[df["cast"].fillna("").str.lower().str.contains(name, regex=False)]
    if r.empty:
        print("\nNo titles found with that actor/actress.")
        return
    print("\n", len(r), "titles found, ranked by popularity:\n")
    show_with_people(r)


def recommend_by_director():
    name = input("Director name (e.g. Christopher Nolan): ").lower().strip()
    if not name:
        print("Please enter a name.")
        return
    r = df[df["director"].fillna("").str.lower().str.contains(name, regex=False)]
    if r.empty:
        print("\nNo titles found for that director.")
        return
    print("\n", len(r), "titles found, ranked by popularity:\n")
    show_with_people(r)


def recommend():
    print("\n--- RECOMMENDATIONS ---")
    print("1. By my preferences (genre, year, country, rating...)")
    print("2. By actor / actress")
    print("3. By director")
    choice = input("Choose 1, 2 or 3: ").strip()
    if choice == "1":
        recommend_by_preferences()
    elif choice == "2":
        recommend_by_actor()
    elif choice == "3":
        recommend_by_director()
    else:
        print("Invalid choice.")


In [8]:
# ---------------- 7. WEB SCRAPING ----------------
def wiki_summary():
    title = input("Title to look up: ").strip().replace(" ", "_")
    headers = {"User-Agent": "NetflixMiniProject/1.0 (student project)"}
    for suffix in ["", "_(film)", "_(TV_series)"]:
        page = requests.get("https://en.wikipedia.org/wiki/" + title + suffix, headers=headers, timeout=10)
        if page.status_code == 200:
            soup = BeautifulSoup(page.text, "html.parser")
            for p in soup.find_all("p"):
                if len(p.get_text()) > 120:
                    print("\n", p.get_text().strip())
                    return
    print("No page found.")

In [ ]:
# ---------------- 8. MENU ----------------
menu = {
    "1": ("Dataset summary & missing values", summary),
    "2": ("Movie vs TV Show count", movie_vs_show),
    "3": ("Top 10 genres", top_genres),
    "4": ("Top countries", top_countries),
    "5": ("Release-year trend", release_trend),
    "6": ("Content ratings", rating_chart),
    "7": ("Additions over time", added_over_time),
    "8": ("Average IMDb rating + distribution", rating_analysis),
    "9": ("Highest-rated movies & shows", highest_rated),
    "10": ("Most popular movies & shows", most_popular),
    "11": ("Top 10 by genre", top_by_genre),
    "12": ("Popular titles from a country", top_by_country),
    "13": ("Recommend me something", recommend),
    "14": ("Wikipedia summary (web scraping)", wiki_summary),
}

while True:
    print("\n===== NETFLIX ANALYZER =====")
    for k, (name, _) in menu.items():
        print(k + ".", name)
    print("0. Exit")
    choice = input("Choose: ").strip()
    if choice == "0":
        print("Bye!")
        break
    elif choice in menu:
        menu[choice][1]()
    else:
        print("Invalid choice.")


===== NETFLIX ANALYZER =====
1. Dataset summary & missing values
2. Movie vs TV Show count
3. Top 10 genres
4. Top countries
5. Release-year trend
6. Content ratings
7. Additions over time
8. Average IMDb rating + distribution
9. Highest-rated movies & shows
10. Most popular movies & shows
11. Top 10 by genre
12. Popular titles from a country
13. Recommend me something
14. Wikipedia summary (web scraping)
0. Exit
